# 01-9_remove_wcs_in_HDUL_ys

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, astropy, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name==version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

In [ ]:
#import sys
#!pip install photutils==1.6

In [1]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        print(f"**** module {pkg} is not installed")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else: 
        print(f"**** module {pkg} is installed")
%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

**** module numpy is installed
**** module pandas is installed
**** module matplotlib is installed
**** module scipy is installed
**** module astropy is installed
**** module photutils is installed
**** module ccdproc is installed
**** module version_information is installed


### import modules

In [3]:
import os, shutil
from glob import glob
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import astropy.units as u
from astropy.stats import sigma_clip
from astropy.io import fits
from ccdproc import combine, ccd_process, CCDData

import ysfitsutilpy as yfu
#import ysphotutilpy as ypu
import ysvisutilpy as yvu

from snuo1mpy import Preprocessor

import astro_utilities
import Python_utilities

plt.rcParams.update({'figure.max_open_warning': 0})

In [4]:
#%%
BASEDIR = Path("/mnt/OBS_data") 
BASEDIR = Path("/mnt/Rdata/CCD_obs") 
#BASEDIR = Path("/mnt/OBS_data") 
#DOINGDIR = Path( BASEDIR/ astro_utilities.CCD_obs_raw_dir)
DOINGDIR = Path( BASEDIR/ astro_utilities.CCD_NEW_dir)
DOINGDIRs = sorted(Python_utilities.getFullnameListOfallsubDirs(DOINGDIR))
#print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))

len(DOINGDIRs):  166


In [6]:

for DOINGDIR in DOINGDIRs :
    # 디렉토리 하나씩 loop...
    print ("Starting...\n{}".format(DOINGDIR))

    DOINGDIR = Path(DOINGDIR)

    #파일 목록을 DataFrame으로 만들어서 이용하자
    summary = yfu.make_summary(DOINGDIR / "*.fit*")
                        #keywords = ["DATE-OBS", "FILTER", "OBJECT", "IMAGETYP"],  # header keywords; actually it is case-insensitive
                        #fname_option = 'name',  # 'file' column will contain only the name of the file (not full path)
                        #output = "{}summary.csv".format(DOINGDIR),
                        #sort_by = "DATE-OBS"  # 'file' column will be sorted based on "DATE-OBS" value in the header
                        #)
    if summary.empty :
        print(f"There is no fits fils in {DOINGDIR}")
        pass
    else : 
        # light frame  만 선택
        #df_light = summary[summary["IMAGETYP"] == "LIGHT"]
        df_light = summary
        #print ("df_light: {}".format(df_light))
        print ("len(df_light): {}".format(len(df_light)))

        for _, row in df_light.iterrows():
            # 파일명 출력
            print (row["file"])
            fpath = Path(row["file"])
            new_fpath = Path(f"{fpath.parents[0]}/{fpath.stem}_clean.fit")
            # fits hedaer 에 있는 wcs 정보를 지운다
            yfu.wcsremove(fpath, 
                        additional_keys=["COMMENT"],
                        verbose=True,
                        output=new_fpath,
                        ccddata=True,
                        overwrite=True)
            if new_fpath.exists() \
                and fpath.exists():
                print("rename", f"{str(new_fpath)}", f"{str(fpath)}")
                #os.rename(f"{str(new_fpath)}", f"{str(fpath)}")
                shutil.move(f"{str(new_fpath)}", f"{str(fpath)}")

Starting...
/mnt/Rdata/CCD_obs/CCD_new_files/QSI683ws_1bin/
No FITS file found.
There is no fits fils in /mnt/Rdata/CCD_obs/CCD_new_files/QSI683ws_1bin
Starting...
/mnt/Rdata/CCD_obs/CCD_new_files/QSI683ws_1bin/Light_FSQ106ED-x73//
No FITS file found.
There is no fits fils in /mnt/Rdata/CCD_obs/CCD_new_files/QSI683ws_1bin/Light_FSQ106ED-x73
Starting...
/mnt/Rdata/CCD_obs/CCD_new_files/QSI683ws_1bin/Light_FSQ106ED-x73/21P_Light_-_2018-09-10_-_FSQ106ED-x73_QSI683ws_-_1bin///
All 68 keywords (guessed from /mnt/Rdata/CCD_obs/CCD_new_files/QSI683ws_1bin/Light_FSQ106ED-x73/21P_Light_-_2018-09-10_-_FSQ106ED-x73_QSI683ws_-_1bin/21P_Light_B_2018-09-10-16-41-60_100sec_FSQ106ED-x73_QSI683ws_-20C_1bin.fit) will be loaded.


ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [8]:
df_light = summary[summary["IMAGETYP"] == "LIGHT"]
print ("df_light: {}".format(df_light))
print ("len(df_light): {}".format(len(df_light)))

df_light:                                                   file  filesize  SIMPLE  \
172  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
173  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
174  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
175  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
176  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
..                                                 ...       ...     ...   
299  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
300  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
301  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
302  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
303  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   

     BITPIX  NAXIS  NAXIS1  NAXIS2 EXTEND    BZERO IMAGETYP  ...  FOCALLEN  \

In [16]:
for _, row in df_light.iterrows():
        # 파일명 출력
        print (row["file"])
        fpath = Path(row["file"])
        new_fpath = Path(f"{fpath.parents[0]}/{fpath.stem}_clean.fit")
        # fits hedaer 에 있는 wcs 정보를 지운다
        yfu.wcsremove(fpath, 
                    additional_keys=["COMMENT"],
                    verbose=True,
                    output=new_fpath,
                    ccddata=True,
                    overwrite=True)
        if new_fpath.exists() \
            and fpath.exists():
            print("rename", f"{str(new_fpath)}", f"{str(fpath)}")
            #os.rename(f"{str(new_fpath)}", f"{str(fpath)}")
            shutil.move(f"{str(new_fpath)}", f"{str(fpath)}")
            
    # except Exception as err :
    #     print("X"*60)
    #     print('{0}'.format(err))

/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/KLEOPATRA_Light_b_2022-10-23-11-59-11_180sec_RiLA600_STX-16803_-30C_2bin.fit
Removed keywords: 
rename /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/KLEOPATRA_Light_b_2022-10-23-11-59-11_180sec_RiLA600_STX-16803_-30C_2bin_clean.fit /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/KLEOPATRA_Light_b_2022-10-23-11-59-11_180sec_RiLA600_STX-16803_-30C_2bin.fit
/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/KLEOPATRA_Light_b_2022-10-23-11-59-11_180sec_RiLA600_STX-16803_-30C_2bin_clean.fit
Removed keywords: 


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/KLEOPATRA_Light_b_2022-10-23-11-59-11_180sec_RiLA600_STX-16803_-30C_2bin_clean.fit'